---
title: Linked-Read Preprocess Summary
subtitle: A look at barcode preprocessing in raw reads
date: "9999-12-31"
edit_url: null
---
**Linked-read type**: Haplotagging (GIH)

This report describes linked-read library performance, as assessed following barcode demultiplexing[^1] from the sample sequences. The values in the boxes below are averages across your samples. 

[^1]:
    Demultiplexing barcodes describes identifying the barcode segments and moving them into the `BX` and `VX` header tags.

In [ ]:
from harpy.report.components import StatsBox, standard_itable
from harpy.report.theme import palette
import os
import polars as pl
from pathlib import Path
pl.Config.set_tbl_hide_column_data_types()


In [ ]:
indir = "Preprocess/reports/data"
d = {}

In [ ]:
BX_infiles = list(Path(indir).glob("*.BXstats"))

d['Sample'] = []
d['Total Reads'] = []
d['Unique Barcodes'] = []
d['Valid Barcodes'] = []
d['Corrected Barcodes'] = []
d['ME Absent'] = []
d['Too Short'] = []

for infile in BX_infiles:
    d['Sample'].append(infile.stem.removesuffix('.BXstats'))
    with open(infile, 'r') as f:
        d['Unique Barcodes'].append(int(f.readline().split()[-1]))
        d['Valid Barcodes'].append(int(f.readline().split()[-1]))
        d['Corrected Barcodes'].append(int(f.readline().split()[-1]))

for smp in d['Sample']:
    infile = os.path.join(indir, f"{smp}.MEstats")
    with open(infile, 'r') as f:
        d['Total Reads'].append(int(f.readline().split()[-1]))
        d['ME Absent'].append(int(f.readline().split()[-1]))
        d['Too Short'].append(int(f.readline().split()[-1]))

data = pl.DataFrame(d)


In [ ]:
avg_valid = data.select(
    (pl.col("Valid Barcodes") / pl.col("Total Reads")).mean()
).item()

avg_corrected = data.select(
    (pl.col("Corrected Barcodes") / pl.col("Total Reads")).mean()
).item()

avg_me_discard = data.select(
    (pl.col("ME Absent") / pl.col("Total Reads")).mean()
).item()

avg_short = data.select(
    (pl.col("Too Short") / pl.col("Total Reads")).mean()
).item()

(
    StatsBox()
    .add(len(d['Sample']), "Samples")
    .add(round(avg_valid*100,1), "% Valid")
    .add(round(avg_corrected*100, 5), "% Corrected")
    .add(round(avg_me_discard*100, 5), "% ME Discard")
    .add(round(avg_short*100, 5), "% Too Short")
    .render()
)

In the table below, `Valid Barcodes` is the sum total valid barcodes for R1 and R2 reads (divide by 2 for valid per read-pair). The `Corrected Barcodes` is the number of reads (R1 + R2) that had at least one unidentified barcode segment that was recovered by `Pheniqs`. The `ME Absent` and `Too Short` columns track the number of read pairs that were removed due to the ME sequence not being found and the reads being too short after trimming barcodes/ME, respectively.

In [ ]:

standard_itable(data, filename = "preprocess")